In [1]:
import duckdb
import pandas as pd
import lightgbm as lgb
import pickle
from sklearn.metrics import mean_absolute_error

## 1. Load Data

In [2]:
con = duckdb.connect('../data/fpl-mentat.duckdb')

In [3]:
df = con.execute("""
select * from fact_ml_features
where avg_minutes_gw3 is not null
""").fetchdf()

In [4]:
# Check target distribution
print(df['target_total_points'].describe())
print(f"% of rows with 0 points: {(df['target_total_points'] == 0).mean():.1%}")

count    24385.000000
mean         1.183104
std          2.405390
min         -3.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         24.000000
Name: target_total_points, dtype: float64
% of rows with 0 points: 62.8%


In [5]:
print(f"Total rows: {len(df)}")
print(f"Gameweeks: {df['gameweek'].min()} to {df['gameweek'].max()}")

Total rows: 24385
Gameweeks: 2 to 32


In [6]:
# Remove players with noisy data- only keep players with meaningful minutes
df = df[df['avg_minutes_gw3'] > 20]

In [7]:
# Identify rolling stat columns to impute
rolling_cols = [c for c in df.columns if c.startswith('avg_') and ('gw3' in c or 'gw5' in c)]

# For each player, calculate their season average for each stat
# using only gameweeks where they actually played
played_df = df[df['avg_minutes_gw3'] > 0]  # only played gameweeks
player_season_avgs = played_df.groupby('player_id')[rolling_cols].mean()

# Impute: where a blank occurred in the rolling window, 
# replace the depressed rolling avg with the season avg
def impute_blank_rows(row):
    # A blank in gw3 window means fixture_cnt_gw3 < 3
    # A blank in gw5 window means fixture_cnt_gw5 < 5
    needs_gw3_impute = row['fixture_cnt_gw3'] < 3
    needs_gw5_impute = row['fixture_cnt_gw5'] < 5
    
    if not (needs_gw3_impute or needs_gw5_impute):
        return row  # no blank, no imputation needed
    
    player_avgs = player_season_avgs.loc[row['player_id']] if row['player_id'] in player_season_avgs.index else None
    
    if player_avgs is None:
        return row
        
    for col in rolling_cols:
        if 'gw3' in col and needs_gw3_impute:
            row[col] = player_avgs[col]
        elif 'gw5' in col and needs_gw5_impute:
            row[col] = player_avgs[col]
    
    return row

df = df.apply(impute_blank_rows, axis=1)

# Cast all feature columns back to numeric
for col in rolling_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Also cast fixture_difficulty specifically
df['fixture_difficulty'] = pd.to_numeric(df['fixture_difficulty'], errors='coerce')

## 2. Exclude features and Target

In [8]:
# Drop these fields, will encode below
EXCLUDE = [
    'player_id', 'gameweek', 'team_id', 'opponent_id',
    'target_total_points', 'is_prediction_row',
    'home_or_away', 'position'
]

In [9]:
# Encode categorical variables
df['home_enc'] = df['home_or_away'].map({'home': 1, 'away': 0})

In [10]:
features = list(dict.fromkeys([c for c in df.columns if c not in EXCLUDE]))
target = 'target_total_points'

In [11]:
print([f for f in features if 'point' in f.lower()])

['avg_total_points_gw3', 'avg_total_points_gw5']


## 3. Train/Val/Test Split

In [12]:
train = df[df['gameweek'] <= 25]
val = df[(df['gameweek'] > 25) & (df['gameweek'] <= 29)]
test = df[df['gameweek'] > 29]

In [13]:
X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]
X_test, y_test = test[features], test[target]

In [14]:
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 6874 | Val: 1347 | Test: 876


## 4. Train Model

In [15]:
lgbmodel = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=62,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [16]:
lgbmodel.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001277 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8893
[LightGBM] [Info] Number of data points in the train set: 6874, number of used features: 64
[LightGBM] [Info] Start training from score 2.758947
Training until validation scores don't improve for 50 rounds
[50]	valid_0's l2: 10.7889
Early stopping, best iteration is:
[25]	valid_0's l2: 10.7116


,boosting_type,'gbdt'
,num_leaves,62
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## 5. Evaluate

In [17]:
val_preds = lgbmodel.predict(X_val)
print(f"Val MAE:  {mean_absolute_error(y_val, val_preds):.3f}")

Val MAE:  2.346


In [18]:
test_preds = lgbmodel.predict(X_test)
print(f"Test MAE: {mean_absolute_error(y_test, test_preds):.3f}")

Test MAE: 2.215


In [19]:
# MAE by position
test['predicted'] = test_preds
test['error'] = abs(test['predicted'] - test[target])
print("\nMAE by position:")
print(test.groupby('position')['error'].mean().sort_values())


MAE by position:
position
3    1.937250
1    2.170391
4    2.320143
2    2.507558
Name: error, dtype: float64


## 6. Feature Importance

In [20]:
importance = pd.DataFrame({
    'feature': features,
    'importance': lgbmodel.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 features:")
print(importance.head(20).to_string(index=False))


Top 20 features:
                                feature  importance
        avg_expected_goals_conceded_gw3          61
                    minutes_variability          60
                     fixture_difficulty          60
        avg_expected_goals_conceded_gw5          54
     avg_expected_goal_involvements_gw3          46
                        avg_minutes_gw3          44
     avg_expected_goal_involvements_gw5          43
                            avg_bps_gw5          43
              team_avg_goals_scored_gw5          43
avg_clearances_blocks_interceptions_gw3          42
                        avg_minutes_gw5          40
         avg_defensive_contribution_gw5          39
                             avg_xG_gw5          39
                         avg_threat_gw3          38
                             form_trend          37
                   avg_total_points_gw5          36
                     avg_creativity_gw5          36
                            avg_bps_gw3       

## 7. Prediction for next gameweek

In [21]:
latest_gw = df['gameweek'].max()

predict_df = con.execute(f"""
    select * from fact_ml_features
    where gameweek = {latest_gw}
""").fetchdf()

In [22]:
predict_df.head()

,player_id,gameweek,next_gw_fixture_cnt,fixture_cnt_gw3,fixture_cnt_gw5,position,team_id,opponent_id,home_or_away,fixture_difficulty,...,opp_avg_goals_scored_gw5,opp_avg_goals_conceded_gw3,opp_avg_goals_conceded_gw5,avg_total_points_gw3,avg_total_points_gw5,form_trend,minutes_variability,had_blank_gw,is_prediction_row,target_total_points
0,9.0,32,1.0,2.0,4.0,2,1,4,home,3,...,0.6,0.666667,0.6,0.000000,0.0,0.000000,0.000000,0,N,0.0
1,61.0,32,1.0,3.0,5.0,3,2,16,away,3,...,1.2,0.666667,1.0,0.000000,0.0,0.000000,0.000000,0,N,0.0
2,73.0,32,2.0,3.0,5.0,2,4,1,away,5,...,2.2,0.333333,0.8,2.666667,1.8,-0.866667,25.851499,0,N,1.0
3,76.0,32,2.0,3.0,5.0,2,4,1,away,5,...,2.2,0.333333,0.8,0.000000,0.0,0.000000,0.000000,0,N,0.0
4,102.0,32,1.0,3.0,5.0,1,5,9,home,3,...,1.6,0.666667,1.0,0.000000,0.0,0.000000,0.000000,0,N,0.0


In [23]:
invalid_players = [399.0,182,421]  # Ederson's player_id

predict_df = predict_df[~predict_df['player_id'].isin(invalid_players)]

predict_df = predict_df.apply(impute_blank_rows, axis=1)

for col in rolling_cols:
    predict_df[col] = pd.to_numeric(predict_df[col], errors='coerce')

predict_df['fixture_difficulty'] = pd.to_numeric(predict_df['fixture_difficulty'], errors='coerce')

In [24]:
# Generate base predictions
predict_df['home_enc'] = predict_df['home_or_away'].map({'home': 1, 'away': 0})

predict_df['predicted_points'] = lgbmodel.predict(predict_df[features])
predict_df['predicted_points'] = predict_df.apply(
    lambda x: x['predicted_points'] * 1.8 if x['next_gw_fixture_cnt'] == 2 else x['predicted_points'],
    axis=1
)

In [37]:
# join player names from dim_players
players = con.execute("""
    select player_id, first_name, second_name, curr_price, ownership_percent
    from dim_players
""").fetchdf()

excluded_player_ids = [
    408
]

recommendations = predict_df[~predict_df['player_id'].isin(excluded_player_ids)].merge(players, on='player_id')[[
    'player_id', 'first_name', 'second_name', 'position', 'curr_price',
    'ownership_percent', 'fixture_difficulty',
    'next_gw_fixture_cnt', 'predicted_points'
]].sort_values('predicted_points', ascending=False)


pos_map = {
    1:'Goalkeeper',
    2:'Defender',
    3:'Midfielder',
    4:'Forward'
}

print("\n--- TOP RECOMMENDATIONS ---")
for position in [1, 2, 3, 4]:
    print(f"\n{pos_map[position]}:")
    print(
        recommendations[recommendations['position'] == position]
        .head(5)
        .to_string(index=False)
    )


--- TOP RECOMMENDATIONS ---

Goalkeeper:
 player_id first_name second_name  position  curr_price ownership_percent  fixture_difficulty  next_gw_fixture_cnt  predicted_points
     139.0       Bart  Verbruggen         1         4.5              10.5                   2                  2.0          5.860542
     341.0       Karl      Darlow         1         3.9               7.4                   4                  2.0          5.195334
     736.0  Gianluigi  Donnarumma         1         5.6               9.5                   3                  2.0          4.927707
      67.0      Đorđe    Petrović         1         4.6               6.2                   5                  2.0          4.599245
     470.0     Martin    Dúbravka         1         4.0              28.5                   3                  2.0          4.262690

Defender:
 player_id first_name second_name  position  curr_price ownership_percent  fixture_difficulty  next_gw_fixture_cnt  predicted_points
     151.0   Jan

In [38]:
# Differential picks - ownership under 10%
differentials = predict_df[~predict_df['player_id'].isin(excluded_player_ids)].merge(players, on='player_id')[[
    'player_id', 'first_name', 'second_name', 'position', 'curr_price',
    'ownership_percent', 'fixture_difficulty',
    'next_gw_fixture_cnt', 'predicted_points'
]]

differentials = differentials[differentials['ownership_percent'].astype(float) < 10]
differentials = differentials.sort_values('predicted_points', ascending=False)

print("\n--- DIFFERENTIAL PICKS (< 10% ownership) ---")
for position in [1, 2, 3, 4]:
    print(f"\n{pos_map[position]}:")
    print(
        differentials[differentials['position'] == position]
        .head(5)
        .to_string(index=False)
    )


--- DIFFERENTIAL PICKS (< 10% ownership) ---

Goalkeeper:
 player_id first_name second_name  position  curr_price ownership_percent  fixture_difficulty  next_gw_fixture_cnt  predicted_points
     341.0       Karl      Darlow         1         3.9               7.4                   4                  2.0          5.195334
     736.0  Gianluigi  Donnarumma         1         5.6               9.5                   3                  2.0          4.927707
      67.0      Đorđe    Petrović         1         4.6               6.2                   5                  2.0          4.599245
     221.0      Filip   Jörgensen         1         4.3               0.2                   4                  2.0          4.108031
     789.0        Max     Merrick         1         4.0               0.0                   4                  2.0          3.822458

Defender:
 player_id first_name second_name  position  curr_price ownership_percent  fixture_difficulty  next_gw_fixture_cnt  predicted_points